# Persistent FAQ indexes with SQLite and Elasticsearch

This notebook loads the Zoomcamp FAQ data and compares two persistent retrieval backends:

- `sqlitesearch` for a lightweight local index stored in a database file;
- Elasticsearch for a service-backed index with field boosting and filtering.

It also verifies that the Elasticsearch index survives a container restart.

## 1. Load and inspect the FAQ data

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

In [ ]:
type(documents)

In [ ]:
type(documents[0])

`documents` is a list of dictionaries. Each dictionary represents one FAQ entry and contains fields such as the course, section, question, and answer.

In [ ]:
documents[100]

In [ ]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

In [ ]:
docs_llm[100]

## 2. Build a course-specific SQLite index

`text_fields` are searched using full-text retrieval, while `keyword_fields` support exact filters such as `course="llm-zoomcamp"`. The `db_path` identifies the persistent SQLite database file.

The index is initially empty. Add each LLM Zoomcamp document, then close the index so the database is flushed to disk.

In [ ]:
import time
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq_llm.db",
)

for doc in docs_llm:
    sqlite_index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    # time.sleep(0.5)  # Optional: slow ingestion for demonstration.

sqlite_index.close()
print("Done. Index saved to faq_llm.db")

In [ ]:
sqlite_index.count()

In [ ]:
sqlite_index

## 3. Query the SQLite index

Begin with a simple query, inspect the search method's signature, and then apply field boosting. Giving the `question` field a higher weight favors FAQ entries whose wording closely matches the query.

In [ ]:
sqlite_index.search("install Docker")

In [ ]:
help(sqlite_index.search)

In [ ]:
results = sqlite_index.search(
    query="How do I submit homework?",
    boost_dict={
        "question": 3.0,
        "section": 1.0,
        "answer": 1.0,
    },
    num_results=5,
)

results

## 4. Inspect the underlying SQLite database

`sqlitesearch` stores each document as serialized JSON. The following cells use standard-library `sqlite3` queries to inspect records, tables, columns, and an individual row. These queries are diagnostic; normal retrieval should use `TextSearchIndex.search()`.

In [ ]:
# Now let's query the SQLite database to see what we have stored.

import sqlite3

with sqlite3.connect("faq_llm.db") as connection:
    connection.row_factory = sqlite3.Row

    records = connection.execute(
        "SELECT * FROM docs LIMIT 10"
        #"SELECT doc_json AS doc_table FROM docs LIMIT 10"    
        #"SELECT * FROM doc_table WHERE question LIKE '%Docker%' OR section LIKE '%Docker%' OR answer LIKE '%Docker%' LIMIT 10"
    ).fetchall()

    for record in records:
        print(type(record))
        print(dict(record))
        #print(record["doc_json"])

In [ ]:
# doc_json looks like a dictionary, but it is currently a string:

print(type(record["doc_json"]))
# <class 'str'>

In [ ]:
# Convert it back into a Python dictionary with json.loads()

import json

for record in records:
    doc_dict = json.loads(record["doc_json"])
    print(type(doc_dict))
    print(doc_dict)

In [ ]:
# Now let's print the individual fields of each document in a more readable format.

for record in records:
    document = json.loads(record["doc_json"])

    print("Document ID:", document["id"])
    print("Course:", document["course"])
    print("Section:", document["section"])
    print("Question:", document["question"])
    print("Answer:", document["answer"])
    print()

In [ ]:
# Now let's query the SQLite database to see what tables we have stored.

with sqlite3.connect("faq_llm.db") as connection:
    tables = connection.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table'"
    ).fetchall()

    for table in tables:
        print(table[0])

In [ ]:
# Now let's query the SQLite database to see what we have stored.

with sqlite3.connect("faq_llm.db") as connection:
    connection.row_factory = sqlite3.Row

    records = connection.execute("""
        SELECT *
        FROM docs
        LIMIT 10
    """).fetchall()

    for record in records:
        print(dict(record))

In [ ]:
# Now let's query the SQLite database to see what columns we have stored.

with sqlite3.connect("faq_llm.db") as connection:
    columns = connection.execute(
        "PRAGMA table_info(docs)"
    ).fetchall()

    for column in columns:
        print(column)

In [ ]:
# You can also examine one particular record by its internal ID:

with sqlite3.connect("faq_llm.db") as connection:
    connection.row_factory = sqlite3.Row

    record = connection.execute(
        "SELECT * FROM docs WHERE id = ?",
        (1,)
    ).fetchone()

    print(dict(record) if record else "Record not found")

## 5. Build an all-course SQLite index

The first index contains only LLM Zoomcamp records. Build a second index from the complete dataset so the same retrieval layer can answer questions about any Zoomcamp. The `course` keyword field can then be used to constrain searches.

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

In [ ]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

for doc in documents:
    sqlite_index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    # time.sleep(0.5)  # Optional: slow ingestion for demonstration.

sqlite_index.close()
print("Done. Index saved to faq.db")

### Connect the persistent index to `RAGBase`

Because `TextSearchIndex` implements the search interface expected by `RAGBase`, the same generation pipeline works with the persisted SQLite backend.

In [ ]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [ ]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

In [ ]:
answer = assistant.rag("When does the next Zoomcamp start?")
print(answer)

In [ ]:
answer = assistant.rag("When does the next Data-Engineering Zoomcamp start?")
print(answer)

In [ ]:
results = sqlite_index.search(
    query="How do I submit homework?",
    boost_dict={
        "question": 3.0,
        "section": 1.0,
        "answer": 1.0,
    },
    num_results=5,
)

results

In [ ]:
sqlite_index.close()

## 6. Build an Elasticsearch index

Elasticsearch provides a service-backed alternative with explicit mappings, bulk indexing, field boosting, and keyword filters. Start the local Elasticsearch service before running the following cells.

In [ ]:
# Now let's try to connect to an Elasticsearch instance and see if we can get some information about it.

from elasticsearch import Elasticsearch

es_client = Elasticsearch("http://localhost:9200")

print(es_client.info())

In [ ]:
# Firstly, we will reload the FAQ data from the JSON file using the `load_faq_data` function 
# from the `ingest` module. This function reads the data and returns a list of documents.

from ingest import load_faq_data

docs_faq = load_faq_data()
print(f"Loaded {len(docs_faq)} documents")

In [ ]:
# Now let's create an Elasticsearch index for all the Zoomcamp FAQ documents

index_name = "faq-zoomcamp"

mapping = {
    "mappings": {
        "properties": {
            "question": {"type": "text"},
            "section": {"type": "text"},
            "answer": {"type": "text"},
            "course": {"type": "keyword"},
        }
    }
}

# Check if the index already exists before creating it.
# In the following code snippet, '**' is used to unpack the mapping dictionary into keyword arguments for the create method. 
# This allows the mapping to be passed as a single argument rather than as separate keyword arguments.

if not es_client.indices.exists(index=index_name):
    es_client.indices.create(
        index=index_name,
        **mapping,
    )

In [ ]:
# Now let's index the Zoomcamp documents into Elasticsearch.

# Bulk indexing is a more efficient way to index multiple documents at once, 
# rather than indexing them one by one.

from elasticsearch.helpers import bulk

# The following code snippet uses a generator expression to create a sequence of actions 
# for the bulk indexing operation.

# The generator expression iterates over the docs_llm list and creates a dictionary for each document.
# Each dictionary contains the index name, document ID, and the document itself as the source.

# The use of a generator expression allows for efficient memory usage, 
# as it generates each action on-the-fly rather than storing them all in memory at once.
# Parentheses '()' are used to create a generator expression, which is a concise way to create an iterator 
# that yields items one at a time.

# Square brackets '[]' would create a list, which would store all items in memory at once.
# Generators are normally single-use, meaning that once they are exhausted, they cannot be reused.

# (...)  = generator expression
# [...]  = list comprehension

actions = (
    {
        "_index": index_name,
        "_id": document["id"],  # Use the document's ID as the Elasticsearch document ID, to avoid duplicates
        "_source": document,
    }
    for document in docs_faq
)

indexed_count, errors = bulk(es_client, actions)

print(f"Indexed {indexed_count} documents")
print(f"Errors: {errors}")

In [ ]:
# Now let's refresh the index to make sure all documents are searchable immediately after indexing.

es_client.indices.refresh(index=index_name)

In [ ]:
# Now let's query the Elasticsearch index to see what we have stored.

response = es_client.search(
    index=index_name,
    size=5,
    query={
        "multi_match": {
            "query": "Can I still join the course?",
            "fields": [
                "question^3",   # Boost the question field to give it more weight in the search results
                "section",
                "answer",
            ],
        }
    },
)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Course:", hit["_source"]["course"])
    print("Question:", hit["_source"]["question"])
    print("Answer:", hit["_source"]["answer"])
    print()

In [ ]:
# Now let's query the Elasticsearch index to see what we have stored, but this time with a filter for the course.

response = es_client.search(
    index=index_name,
    size=5,
    query={
        "bool": {
            "must": {
                "multi_match": {
                    "query": "homework submission",
                    "fields": [
                        "question^3",
                        "section",
                        "answer",
                    ],
                }
            },
            "filter": {
                "term": {
                    "course": "llm-zoomcamp"
                }
            },
        }
    },
)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Course:", hit["_source"]["course"])
    print("Question:", hit["_source"]["question"])
    print("Answer:", hit["_source"]["answer"])
    print()

In [ ]:
es_client.close()

## 7. Verify Elasticsearch persistence

Restart the container and run the same search without recreating or re-ingesting the index. If the query still returns results, the Docker volume has preserved the Elasticsearch data.

Persistence is enabled by mounting a named Docker volume in `docker-compose.yml`:

```yaml
services:
  elasticsearch:
    volumes:
      - elasticsearch-data:/usr/share/elasticsearch/data

volumes:
  elasticsearch-data:
```


In [ ]:
# Restart the Elasticsearch service

!docker compose restart elasticsearch

In [ ]:
# Now let's try to connect to an Elasticsearch instance and see if we can get some information about it.

from elasticsearch import Elasticsearch

es_client = Elasticsearch("http://localhost:9200")

assert es_client.ping()
print("Connected")

In [ ]:
# Now let's query the Elasticsearch index to see if we can retrieve a specific document by its ID.

index_name = "faq-zoomcamp"

response = es_client.search(
    index=index_name,
    size=5,
    query={
        "multi_match": {
            "query": "Can I still join the course?",
            "fields": [
                "question^3",   # Boost the question field to give it more weight in the search results
                "section",
                "answer",
            ],
        }
    },
)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Course:", hit["_source"]["course"])
    print("Question:", hit["_source"]["question"])
    print("Answer:", hit["_source"]["answer"])
    print()

In [ ]:
es_client.close()